# 📊 Pipeline de Dados com AWS S3, Spark e Glue

## Aula Prática e Introdutória

Bem-vindo! Neste notebook, você aprenderá a construir um **pipeline de dados completo** na AWS.

### 🎯 O que você vai fazer:
1. ✅ Estabelecer conexão com a Conta AWS Academy Lab
2. ✅ Criar Bucket, Pastas e Subpastas
3. ✅ Ingestão de arquivos.csv
4. ✅ Criação de tabela Parquet via Spark

### ⏱️ Tempo estimado: 1 hora

---

## 📋 Pré-requisitos

Antes de começar, certifique-se que tem:

```bash
# Instalar bibliotecas necessárias
pip install boto3 pandas pyarrow pyspark
```

✅ **Credenciais AWS configuradas**
```bash
aws configure
```

✅ **Variáveis de ambiente** (opcional, mais seguro)
```bash
export AWS_ACCESS_KEY_ID="seu_access_key"
export AWS_SECRET_ACCESS_KEY="seu_secret_key"
export AWS_DEFAULT_REGION="us-east-1"
```

In [3]:
import subprocess
import sys

# ============================================================
# Instalar dependências do projeto
# ============================================================

print("📦 Instalando dependências necessárias...\n")

required_packages = {
    "boto3": "1.28.85",
    "pandas": "2.1.3",
    "pyarrow": "14.0.1",
    "pyspark": "3.5.0"
}

failed_packages = []

for package, version in required_packages.items():
    try:
        __import__(package)
        print(f"   ✅ {package:<20} já instalado")
    except ImportError:
        print(f"   ⏳ {package:<20} instalando...", end=" ")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}", "-q"])
            print("✅")
        except Exception as e:
            print(f"❌ Erro: {str(e)}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n⚠️  {len(failed_packages)} pacote(s) falharam ao instalar:")
    for pkg in failed_packages:
        print(f"   - {pkg}")
    print(f"\nTente instalar manualmente:")
    print(f"   pip install -r requirements.txt\n")
else:
    print(f"\n✅ Todas as dependências estão prontas!\n")


📦 Instalando dependências necessárias...

   ✅ boto3                já instalado
   ✅ pandas               já instalado
   ✅ pyarrow              já instalado
   ✅ pyspark              já instalado

✅ Todas as dependências estão prontas!



## 🪣 Célula 2: Criar Bucket e Pastas no S3

Vamos criar o bucket e sua estrutura de diretórios:

In [4]:
# ============================================================
# Configuração Global - Variáveis Reutilizáveis
# ============================================================

from pathlib import Path
import boto3
from botocore.exceptions import ClientError
import sys

# Região da AWS
AWS_REGION = "us-east-1"

# Nome do bucket (ATENÇÃO: precisa ser único em toda a AWS)
BUCKET_NAME = "129264105405-lab"

# Pasta local com os arquivos CSV
LOCAL_FOLDER = Path("../03_arquivos_exemplo").resolve()

# Mapping de arquivos -> prefixos no S3
S3_PREFIXES = {
    "clientes.csv": "data-input/bronze/base-clientes/",
    "vendas.csv": "data-input/bronze/base-vendas/",
}

# Pastas que serão criadas no bucket
FOLDERS = [
    "data-input/bronze/base-clientes/",
    "data-input/bronze/base-vendas/",
    "data-output/bronze/base-clientes/",
    "data-output/silver/base-clientes/",
    "data-output/gold/base-clientes/",
    "data-output/bronze/base-vendas/",
    "data-output/silver/base-vendas/",
    "data-output/gold/base-vendas/",
    "athena-logs/"
]

# Inicializar cliente S3
s3 = boto3.client("s3", region_name=AWS_REGION)

print("✅ Configuração carregada com sucesso!")
print(f"   Bucket: {BUCKET_NAME}")
print(f"   Região: {AWS_REGION}")
print(f"   Arquivos locais: {LOCAL_FOLDER}")


✅ Configuração carregada com sucesso!
   Bucket: 129264105405-lab
   Região: us-east-1
   Arquivos locais: C:\Users\rmfig\OneDrive\Desktop\FIAP-PosTech\Materiais\Pipeline de Dados - Camada Bronze\03_arquivos_exemplo


In [ ]:
# ============================================================
# PASSO 1 - Criar Bucket e Folders no Amazon S3 (Com Tratamento de Erros)
# ============================================================

print("📋 Validando credenciais AWS...")
try:
    identity = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()
    print(f"   ✅ Autenticado como: {identity['Account']}")
except Exception as e:
    print(f"   ❌ Erro de autenticação: {str(e)}")
    sys.exit(1)

# Criar bucket
print(f"\n🪣 Criando bucket: s3://{BUCKET_NAME}")
try:
    if AWS_REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": AWS_REGION}
        )
    print(f"   ✅ Bucket criado com sucesso")
except ClientError as e:
    if e.response["Error"]["Code"] == "BucketAlreadyOwnedByYou":
        print(f"   ℹ️  Bucket já existe e é seu")
    elif e.response["Error"]["Code"] == "BucketAlreadyExists":
        print(f"   ❌ Erro: Bucket já existe (propriedade de outro usuário)")
        sys.exit(1)
    else:
        print(f"   ❌ Erro ao criar bucket: {str(e)}")
        sys.exit(1)

# Criar folders
print("\n📁 Criando estrutura de pastas...")
failed_folders = []

for folder in FOLDERS:
    try:
        s3.put_object(Bucket=BUCKET_NAME, Key=folder)
        print(f"   ✅ {folder}")
    except ClientError as e:
        print(f"   ❌ {folder} - Erro: {str(e)}")
        failed_folders.append(folder)

if failed_folders:
    print(f"\n⚠️  {len(failed_folders)} pasta(s) falharam ao criar")
else:
    print(f"\n✅ Todas as {len(FOLDERS)} pastas criadas com sucesso!")

# Listar estrutura criada
print(f"\n📊 Estrutura do bucket s3://{BUCKET_NAME}:")
try:
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET_NAME)
    
    objetos = []
    for page in pages:
        objetos.extend(page.get("Contents", []))
    
    if objetos:
        for obj in sorted(objetos, key=lambda x: x["Key"]):
            print(f"   📄 {obj['Key']}")
    else:
        print("   (vazio)")
except ClientError as e:
    print(f"   ❌ Erro ao listar: {str(e)}")


## ✅ Célula 3: Upload de arquivos para S3

Vamos verificar que os arquivos foram enviados corretamente:

In [ ]:
# ============================================================
# PASSO 2 - Upload de Arquivos CSV para S3 (Com Validações)
# ============================================================

print("📤 Iniciando upload de arquivos para S3\n")

# Verificar se pasta local existe
if not LOCAL_FOLDER.exists():
    print(f"❌ Erro: Pasta não encontrada: {LOCAL_FOLDER}")
    sys.exit(1)

print(f"📁 Procurando arquivos CSV em: {LOCAL_FOLDER}")

# Encontrar arquivos CSV
csv_files = sorted(list(LOCAL_FOLDER.glob("*.csv")))

if not csv_files:
    print(f"   ⚠️  Nenhum arquivo CSV encontrado")
else:
    print(f"   ✅ Encontrados {len(csv_files)} arquivo(s)\n")
    
    uploaded_files = []
    failed_files = []
    
    for file_path in csv_files:
        try:
            # Validações
            if not file_path.is_file():
                print(f"   ⚠️  {file_path.name} - não é um arquivo válido")
                continue
            
            if file_path.stat().st_size == 0:
                print(f"   ⚠️  {file_path.name} - arquivo vazio, pulando")
                continue
            
            # Determinar prefixo S3
            prefix = S3_PREFIXES.get(file_path.name, "data-output/bronze/outros/")
            s3_key = f"{prefix}{file_path.name}"
            
            # Upload
            file_size_mb = file_path.stat().st_size / (1024 * 1024)
            print(f"   ⏳ {file_path.name} ({file_size_mb:.2f} MB)...", end=" ")
            
            s3.upload_file(str(file_path), BUCKET_NAME, s3_key)
            
            print(f"✅ → s3://{BUCKET_NAME}/{s3_key}")
            uploaded_files.append((file_path.name, s3_key))
            
        except FileNotFoundError:
            print(f"   ❌ {file_path.name} - arquivo não encontrado")
            failed_files.append(file_path.name)
        except ClientError as e:
            print(f"   ❌ {file_path.name} - Erro AWS: {str(e)}")
            failed_files.append(file_path.name)
        except Exception as e:
            print(f"   ❌ {file_path.name} - Erro: {str(e)}")
            failed_files.append(file_path.name)
    
    # Resumo
    print(f"\n{'='*60}")
    print(f"📊 RESUMO DO UPLOAD")
    print(f"{'='*60}")
    print(f"   ✅ Sucesso: {len(uploaded_files)}/{len(csv_files)} arquivo(s)")
    
    if failed_files:
        print(f"   ❌ Falhas:  {len(failed_files)}")
        for file_name in failed_files:
            print(f"      - {file_name}")
    
    print(f"{'='*60}\n")


In [ ]:
# ============================================================
# PASSO 3 - Listar Arquivos no Bucket S3 (Com Formatação)
# ============================================================

print(f"📊 Listando objetos do bucket: s3://{BUCKET_NAME}\n")

try:
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET_NAME)
    
    objetos = []
    for page in pages:
        objetos.extend(page.get("Contents", []))
    
    if not objetos:
        print("   ℹ️  Nenhum arquivo encontrado")
    else:
        # Agrupar por pasta
        objetos_por_pasta = {}
        tamanho_total = 0
        
        for obj in objetos:
            key = obj["Key"]
            pasta = key.rsplit("/", 1)[0] if "/" in key else "root"
            
            if pasta not in objetos_por_pasta:
                objetos_por_pasta[pasta] = []
            
            objetos_por_pasta[pasta].append({
                "nome": key.rsplit("/", 1)[1] if "/" in key else key,
                "tamanho": obj["Size"],
                "chave_completa": key
            })
            
            tamanho_total += obj["Size"]
        
        # Exibir por pasta
        for pasta in sorted(objetos_por_pasta.keys()):
            print(f"📁 {pasta}/")
            for arquivo in objetos_por_pasta[pasta]:
                tamanho_kb = arquivo["tamanho"] / 1024
                tamanho_mb = arquivo["tamanho"] / (1024 * 1024)
                
                if tamanho_mb > 1:
                    tamanho_str = f"{tamanho_mb:.2f} MB"
                else:
                    tamanho_str = f"{tamanho_kb:.2f} KB"
                
                print(f"   📄 {arquivo['nome']:<30} ({tamanho_str})")
        
        # Sumário
        print(f"\n{'='*60}")
        print(f"📈 SUMÁRIO")
        print(f"{'='*60}")
        print(f"   Total de objetos: {len(objetos)}")
        print(f"   Tamanho total: {tamanho_total / (1024 * 1024):.2f} MB")
        print(f"{'='*60}\n")

except ClientError as e:
    print(f"❌ Erro ao listar objetos: {str(e)}")


## 🚀 Glue Job: Script Automático (Produção)

### O que é um Glue Job?

Um **AWS Glue Job** é um serviço gerenciado que executa scripts ETL (Extract, Transform, Load) de forma:
- ✅ **Escalável**: Spark distribuído
- ✅ **Automatizado**: Pode ser agendado ou disparado por eventos
- ✅ **Monitorado**: Logs no CloudWatch
- ✅ **Sem servidor**: Você só paga pelo que usa

### Fluxo: Notebook → Glue Job

```
Develop no Notebook → Gerar Script .py → Upload S3 → Criar Glue Job
```

Vamos fazer isso agora!


In [15]:
# ============================================================
# PASSO 4 - Upload do Script Glue Job para S3
# ============================================================

print("📤 Fazendo upload do script Glue Job para o bucket de assets do Glue...\n")

import os

# Caminho do script na pasta 04_scripts_python
script_path = str((Path.cwd().parent / "04_scripts_python" / "glue-base-clientes-bronze.py").resolve())

print(f"   📍 Procurando script em: {script_path}")

# Validar se o arquivo existe
if not os.path.exists(script_path):
    print(f"   ❌ Erro: Script não encontrado em {script_path}")
    print(f"   💡 Certifique-se de que o arquivo 'glue-base-clientes-bronze.py' existe na pasta '04_scripts_python'")
    sys.exit(1)

print(f"   ✅ Script encontrado\n")

try:
    account_id = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()["Account"]
    glue_assets_bucket = f"aws-glue-assets-{account_id}-{AWS_REGION}"
    script_s3_key = "scripts/glue-base-clientes-bronze.py"

    print(f"   📦 Bucket do Glue: s3://{glue_assets_bucket}")
    print(f"   📤 Fazendo upload: {script_path}")

    s3.upload_file(
        script_path,
        glue_assets_bucket,
        script_s3_key
    )

    script_s3_uri = f"s3://{glue_assets_bucket}/{script_s3_key}"
    print(f"   ✅ Script enviado com sucesso!")
    print(f"   📍 Localização: {script_s3_uri}\n")

    response = s3.head_object(Bucket=glue_assets_bucket, Key=script_s3_key)
    print(f"   📊 Detalhes:")
    print(f"      Tamanho: {response['ContentLength'] / 1024:.2f} KB")
    print(f"      Last Modified: {response['LastModified']}\n")

except ClientError as e:
    print(f"❌ Erro ao fazer upload: {str(e)}\n")
    script_s3_uri = None
except Exception as e:
    print(f"❌ Erro inesperado: {str(e)}\n")
    script_s3_uri = None

print(f"{'='*70}")
print("✅ Script pronto para ser usado em Glue Job!\n")

📤 Fazendo upload do script Glue Job para o bucket de assets do Glue...

   📍 Procurando script em: C:\Users\rmfig\OneDrive\Desktop\FIAP-PosTech\Materiais\Pipeline de Dados - Camada Bronze\04_scripts_python\glue-base-clientes-bronze.py
   ✅ Script encontrado

   📦 Bucket do Glue: s3://aws-glue-assets-129264105405-us-east-1
   📤 Fazendo upload: C:\Users\rmfig\OneDrive\Desktop\FIAP-PosTech\Materiais\Pipeline de Dados - Camada Bronze\04_scripts_python\glue-base-clientes-bronze.py
   ✅ Script enviado com sucesso!
   📍 Localização: s3://aws-glue-assets-129264105405-us-east-1/scripts/glue-base-clientes-bronze.py

   📊 Detalhes:
      Tamanho: 4.58 KB
      Last Modified: 2026-09-01 03:11:47+00:00

✅ Script pronto para ser usado em Glue Job!



In [14]:
# ============================================================
# PASSO 5 - Criar Glue Job via Boto3 (Sem Terraform)
# ============================================================

print("🏗️  Criando Glue Job no AWS Glue...\n")

# Inicializar cliente Glue e IAM
glue_client = boto3.client('glue', region_name=AWS_REGION)
iam_client = boto3.client('iam', region_name=AWS_REGION)

# O script do job deve ficar no bucket de assets do Glue, e não no bucket do projeto
account_id = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()["Account"]
glue_assets_bucket = f"aws-glue-assets-{account_id}-{AWS_REGION}"

JOB_NAME = "glue-base-clientes-bronze"
ROLE_NAME = "LabRole"  # Role padrão do AWS Academy Lab
script_s3_uri = f"s3://{glue_assets_bucket}/scripts/glue-base-clientes-bronze.py"

print(f"📋 Configuração:")
print(f"   Job Name: {JOB_NAME}")
print(f"   Script S3: {script_s3_uri}")
print(f"   Bucket do Glue: {glue_assets_bucket}")
print(f"   Role: {ROLE_NAME}\n")

# ============================================================
# PASSO 5.1 - Obter ARN da LabRole (AWS Academy)
# ============================================================

print(f"1️⃣  Obtendo ARN da IAM Role '{ROLE_NAME}'...\n")

try:
    role_response = iam_client.get_role(RoleName=ROLE_NAME)
    role_arn = role_response['Role']['Arn']
    print(f"   ✅ Role encontrada!")
    print(f"   📍 ARN: {role_arn}\n")
except Exception as e:
    print(f"   ❌ Erro ao obter Role: {str(e)}\n")
    print(f"   💡 Certifique-se de que a Role '{ROLE_NAME}' existe no AWS Academy Lab\n")
    sys.exit(1)

# ============================================================
# PASSO 5.2 - Criar ou Atualizar Glue Job
# ============================================================

print(f"2️⃣  Criando/Atualizando Glue Job...\n")

try:
    # Verificar se o job já existe
    job_exists = False
    try:
        glue_client.get_job(JobName=JOB_NAME)
        print(f"   ℹ️  Job já existe. Atualizando...\n")
        job_exists = True
    except glue_client.exceptions.EntityNotFoundException:
        job_exists = False

    # Configuração da criação (pode incluir Tags)
    create_job_config = {
        'Name': JOB_NAME,
        'Role': role_arn,
        'Command': {
            'Name': 'glueetl',
            'ScriptLocation': script_s3_uri,
            'PythonVersion': '3'
        },
        'DefaultArguments': {
            '--job-bookmark-option': 'job-bookmark-enable',
            '--enable-spark-ui': 'true',
            '--spark-event-logs-path': f's3://{BUCKET_NAME}/spark-logs/',
            '--enable-glue-datacatalog': 'true',
            '--BUCKET_NAME': BUCKET_NAME
        },
        'GlueVersion': '5.1',
        'WorkerType': 'G.1X',
        'NumberOfWorkers': 2,
        'Timeout': 30,
        'MaxRetries': 0,
        'Description': 'Pipeline: Lê CSVs do S3 e converte para Parquet com catalogação no Glue',
        'Tags': {
            'Environment': 'Development',
            'Project': 'Pipeline-Dados-AWS'
        }
    }

    # Configuração da atualização (sem Tags e sem Name)
    update_job_config = {
        'Role': role_arn,
        'Command': {
            'Name': 'glueetl',
            'ScriptLocation': script_s3_uri,
            'PythonVersion': '3'
        },
        'DefaultArguments': {
            '--job-bookmark-option': 'job-bookmark-enable',
            '--enable-spark-ui': 'true',
            '--spark-event-logs-path': f's3://{BUCKET_NAME}/spark-logs/',
            '--enable-glue-datacatalog': 'true',
            '--BUCKET_NAME': BUCKET_NAME
        },
        'GlueVersion': '5.1',
        'WorkerType': 'G.1X',
        'NumberOfWorkers': 2,
        'Timeout': 30,
        'MaxRetries': 0,
        'Description': 'Pipeline: Lê CSVs do S3 e converte para Parquet com catalogação no Glue'
    }

    if job_exists:
        # API do Glue exige JobName no topo e JobUpdate no corpo
        glue_client.update_job(
            JobName=JOB_NAME,
            JobUpdate=update_job_config
        )
        print(f"   ✅ Job atualizado com sucesso!")
    else:
        glue_client.create_job(**create_job_config)
        print(f"   ✅ Job criado com sucesso!")

    print(f"\n   📊 Detalhes do Job:")
    print(f"      Nome: {JOB_NAME}")
    print(f"      Script: {script_s3_uri}")
    print(f"      Tipo Worker: G.1X")
    print(f"      Número de Workers: 2")
    print(f"      GlueVersion: 5.1")
    print(f"      Timeout: 30 minutos")
    print(f"      Role: {role_arn}\n")

except Exception as e:
    print(f"   ❌ Erro ao criar/atualizar job: {str(e)}\n")
    import traceback
    traceback.print_exc()

print(f"{'='*70}")
print("✅ Glue Job pronto!\n")

🏗️  Criando Glue Job no AWS Glue...

📋 Configuração:
   Job Name: glue-base-clientes-bronze
   Script S3: s3://aws-glue-assets-129264105405-us-east-1/scripts/glue-base-clientes-bronze.py
   Bucket do Glue: aws-glue-assets-129264105405-us-east-1
   Role: LabRole

1️⃣  Obtendo ARN da IAM Role 'LabRole'...

   ✅ Role encontrada!
   📍 ARN: arn:aws:iam::129264105405:role/LabRole

2️⃣  Criando/Atualizando Glue Job...

   ℹ️  Job já existe. Atualizando...

   ✅ Job atualizado com sucesso!

   📊 Detalhes do Job:
      Nome: glue-base-clientes-bronze
      Script: s3://aws-glue-assets-129264105405-us-east-1/scripts/glue-base-clientes-bronze.py
      Tipo Worker: G.1X
      Número de Workers: 2
      GlueVersion: 5.1
      Timeout: 30 minutos
      Role: arn:aws:iam::129264105405:role/LabRole

✅ Glue Job pronto!



## ✅ Célula 4: Executar Glue Job

Agora vamos executar o Glue Job que foi criado e monitorar seu progresso em tempo real:


In [10]:
# ============================================================
# PASSO 6 - Executar Glue Job (Run Job)
# ============================================================

import time
from datetime import datetime

# Garantir que glue_client está definido
if 'glue_client' not in locals():
    print("⚠️  Inicializando clientes AWS...")
    glue_client = boto3.client('glue', region_name=AWS_REGION)
    iam_client = boto3.client('iam', region_name=AWS_REGION)
    print("✅ Clientes AWS inicializados\n")

print("🚀 Executando Glue Job...\n")

JOB_NAME = "glue-base-clientes-bronze"

try:
    # Executar o Glue Job
    print(f"⏳ Iniciando job: {JOB_NAME}...\n")
    
    response = glue_client.start_job_run(
        JobName=JOB_NAME,
        Arguments={
            '--BUCKET_NAME': BUCKET_NAME
        }
    )
    
    job_run_id = response['JobRunId']
    print(f"   ✅ Job iniciado com sucesso!")
    print(f"   📍 Job Run ID: {job_run_id}")
    print(f"   ⏰ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # ============================================================
    # PASSO 6.1 - Monitorar o progresso do Job
    # ============================================================
    
    print(f"{'='*70}")
    print("📊 MONITORANDO PROGRESSO DO JOB")
    print(f"{'='*70}\n")
    
    max_wait_time = 600  # 10 minutos
    check_interval = 5   # Verificar a cada 5 segundos
    elapsed_time = 0
    last_status = None
    
    while elapsed_time < max_wait_time:
        try:
            # Obter status da execução do job
            job_run = glue_client.get_job_run(
                JobName=JOB_NAME,
                RunId=job_run_id
            )
            
            job_status = job_run['JobRun']
            state = job_status.get('JobRunState', 'UNKNOWN')
            
            # Mostrar atualizações apenas quando o status muda
            if state != last_status:
                elapsed_str = f"{elapsed_time//60}m {elapsed_time%60}s"
                
                # Mapear estado para ícone
                state_icon = {
                    'STARTING': '⏳',
                    'RUNNING': '🔄',
                    'SUCCEEDED': '✅',
                    'FAILED': '❌',
                    'STOPPED': '⛔',
                    'TIMEOUT': '⏰',
                    'ERROR': '❌'
                }.get(state, '❓')
                
                print(f"{state_icon} {state:<15} | Tempo decorrido: {elapsed_str}")
                last_status = state
                
                # Se tiver progresso, exibir
                if 'ExecutionTime' in job_status:
                    print(f"   Execution Time: {job_status['ExecutionTime']} ms")
                
                if 'PredictedTime' in job_status and job_status['PredictedTime'] > 0:
                    print(f"   Predicted Time: {job_status['PredictedTime']} ms")
            
            # Verificar se o job terminou
            if state in ['SUCCEEDED', 'FAILED', 'STOPPED', 'TIMEOUT', 'ERROR']:
                print(f"\n{'='*70}")
                print("📋 RESULTADO DO JOB")
                print(f"{'='*70}\n")
                
                # Detalhes finais
                print(f"   Status Final: {state}")
                print(f"   Run ID: {job_run_id}")
                print(f"   Start Time: {job_status.get('StartedOn', 'N/A')}")
                print(f"   End Time: {job_status.get('CompletedOn', 'N/A')}")
                print(f"   Execution Time: {job_status.get('ExecutionTime', 0)} ms")
                
                if state == 'SUCCEEDED':
                    print(f"\n   ✅ Job completado com sucesso!\n")
                else:
                    print(f"\n   ❌ Job finalizou com status: {state}\n")
                    if 'ErrorMessage' in job_status:
                        print(f"   Error: {job_status['ErrorMessage']}\n")
                
                break
            
            time.sleep(check_interval)
            elapsed_time += check_interval
            
        except Exception as e:
            print(f"   ❌ Erro ao verificar status: {str(e)}")
            break
    
    if elapsed_time >= max_wait_time:
        print(f"\n⏰ Timeout atingido ({max_wait_time} segundos)")
        print(f"   O job pode estar ainda em execução. Verifique no AWS Console.\n")
    
    # ============================================================
    # PASSO 6.2 - Exibir link do AWS Console
    # ============================================================
    
    print(f"{'='*70}")
    print("🔗 LINKS ÚTEIS")
    print(f"{'='*70}\n")
    
    console_url = f"https://console.aws.amazon.com/glue/home?region={AWS_REGION}#/jobs/view/{JOB_NAME}"
    print(f"   📍 Monitorar job no Console:")
    print(f"   {console_url}\n")
    
    cloudwatch_url = f"https://console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}#logsV2:log-groups/aws-glue/jobs/{JOB_NAME}"
    print(f"   📊 Logs do CloudWatch:")
    print(f"   {cloudwatch_url}\n")
    
except Exception as e:
    print(f"❌ Erro ao executar job: {str(e)}\n")
    import traceback
    traceback.print_exc()

print(f"{'='*70}")
print("✅ Execução do Glue Job finalizada!\n")

🚀 Executando Glue Job...

⏳ Iniciando job: glue-base-clientes-bronze...

   ✅ Job iniciado com sucesso!
   📍 Job Run ID: jr_3789a28d782b34786cf219296cfd3032d3f80fb99caba5d5023a5c4587a4447b
   ⏰ Timestamp: 2026-09-01 00:01:12

📊 MONITORANDO PROGRESSO DO JOB

🔄 RUNNING         | Tempo decorrido: 0m 0s
   Execution Time: 0 ms
❌ FAILED          | Tempo decorrido: 1m 20s
   Execution Time: 69 ms

📋 RESULTADO DO JOB

   Status Final: FAILED
   Run ID: jr_3789a28d782b34786cf219296cfd3032d3f80fb99caba5d5023a5c4587a4447b
   Start Time: 2026-09-01 00:00:50.622000-03:00
   End Time: 2026-09-01 00:02:13.461000-03:00
   Execution Time: 69 ms

   ❌ Job finalizou com status: FAILED

   Error: AnalysisException: [ALL_PARTITION_COLUMNS_NOT_ALLOWED] Cannot use all columns for partition columns.

🔗 LINKS ÚTEIS

   📍 Monitorar job no Console:
   https://console.aws.amazon.com/glue/home?region=us-east-1#/jobs/view/glue-base-clientes-bronze

   📊 Logs do CloudWatch:
   https://console.aws.amazon.com/cloudwa